In [ ]:
from tqdm.auto import tqdm

import torch
from torch import nn
from torch import optim
from torch.utils.data import DataLoader
from torchvision import datasets
import torchvision.transforms as Transforms

from lion_pytorch import Lion

In [ ]:
class MuonSGD:
    def __init__(self, muon_params, sgd_params, lr):
        self.muon = optim.Muon(muon_params, lr=lr) if muon_params else None
        self.sgd = optim.SGD(sgd_params, lr=lr) if sgd_params else None

    def zero_grad(self):
        if self.muon:
            self.muon.zero_grad()
        if self.sgd:
            self.sgd.zero_grad()

    def step(self, closure=None):
        if self.muon:
            self.muon.step(closure)
        if self.sgd:
            self.sgd.step()


def get_optimizer(params, name, lr):
    name = name.lower()
    if name == "gd":
        return optim.SGD(params, lr=lr)
    if name == "sgd":
        return optim.SGD(params, lr=lr)
    if name == "adam":
        return optim.Adam(params, lr=lr)
    if name == "lion":
        return Lion(params, lr=lr)
    if name == "muon":
        params = list(params)
        muon_params = [p for p in params if p.ndim == 2]
        sgd_params = [p for p in params if p.ndim != 2]
        return MuonSGD(muon_params, sgd_params, lr)
    raise ValueError(f"Unknown optimizer: {name}")

In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden_sizes=(256, 128)):
        super().__init__()
        layers = [nn.Flatten(), nn.Linear(28 * 28, hidden_sizes[0]), nn.ReLU()]
        for i in range(len(hidden_sizes) - 1):
            layers += [nn.Linear(hidden_sizes[i], hidden_sizes[i + 1]), nn.ReLU()]
        layers.append(nn.Linear(hidden_sizes[-1], 10))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [ ]:
def get_loaders(batch_size):
    transform = Transforms.Compose([Transforms.ToTensor(), Transforms.Normalize((0.5,), (0.5,))])
    train = datasets.FashionMNIST("data", train=True, download=True, transform=transform)
    test = datasets.FashionMNIST("data", train=False, download=True, transform=transform)
    return (
        DataLoader(train, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True),
        DataLoader(test, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True),
    )

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total, loss_sum = 0, 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += y.size(0)
        loss_sum += loss.item() * y.size(0)
    return loss_sum / total


def eval_model(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            correct += (out.argmax(dim=1) == y).sum().item()
            total += y.size(0)
    return correct / total

In [ ]:
opt = "muon"
batch_size = 128
lr = 1e-3
epochs = 10

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_loader, test_loader = get_loaders(batch_size if opt != "gd" else len(datasets.FashionMNIST("data", train=True, download=True)))
model = MLP().to(device)
optimizer = get_optimizer(model.parameters(), opt, lr)
criterion = nn.CrossEntropyLoss()

In [ ]:
for epoch in tqdm(range(epochs)):
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    test_acc = eval_model(model, test_loader, device)
    print(f"epoch={epoch+1} loss={train_loss:.4f} test_acc={test_acc:.4f}")